# NFL Prediction



In [2]:
# Install libraries
!pip install -q optuna catboost xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import rankdata
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Libraries loaded.")

Libraries loaded.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# *** UPDATE THIS PATH to match where your notebook is stored ***
%cd "/content/drive/MyDrive/GCI/competition"

/content/drive/MyDrive/GCI/competition


In [6]:
WORK_DIR = Path.cwd()
PATH = WORK_DIR / 'input'

for f in ['train.csv', 'test.csv', 'sample_submission.csv']:
    status = '✓' if (PATH / f).exists() else '✗ MISSING'
    print(f'{status}  {f}')

✓  train.csv
✓  test.csv
✓  sample_submission.csv


In [7]:
train = pd.read_csv(PATH / 'train.csv')
test  = pd.read_csv(PATH / 'test.csv')
sample_sub = pd.read_csv(PATH / 'sample_submission.csv')

train_raw = train.copy()
test_raw  = test.copy()

TARGET = 'Drafted'
ID_COL = 'Id'
HIGH_CARDINALITY_COL = 'School'

NUMERIC_MISSING_COLS = [
    'Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
    'Broad_Jump', 'Agility_3cone', 'Shuttle',
]
CATEGORICAL_COLS = ['Player_Type', 'Position_Type', 'Position']

print(f'train: {train.shape}, test: {test.shape}')
print(f'Draft rate: {train[TARGET].mean():.1%}')

train: (2781, 16), test: (696, 15)
Draft rate: 64.8%


In [8]:
# ── Feature Engineering (Tutorials 1 & 2) ─────────────────────────────────
def prepare_features(train_raw, test_raw):
    train = train_raw.copy()
    test  = test_raw.copy()

    # Tutorial 1: BMI + School count encoding
    train['BMI'] = train['Weight'] / (train['Height'] ** 2)
    test['BMI']  = test['Weight']  / (test['Height']  ** 2)

    school_counts = train[HIGH_CARDINALITY_COL].value_counts()
    train['School_CE'] = train[HIGH_CARDINALITY_COL].map(school_counts).fillna(0)
    test['School_CE']  = test[HIGH_CARDINALITY_COL].map(school_counts).fillna(0)

    # Missing indicators (before imputation)
    for c in NUMERIC_MISSING_COLS:
        train[f'{c}_missing'] = train[c].isna().astype(int)
        test[f'{c}_missing']  = test[c].isna().astype(int)

    # Mean imputation using train mean only
    for c in NUMERIC_MISSING_COLS:
        mean_value = train[c].mean()
        train[c] = train[c].fillna(mean_value)
        test[c]  = test[c].fillna(mean_value)

    # Tutorial 2: Domain features
    train['SPEED_SCORE'] = (train['Weight'] * 200.0) / (train['Sprint_40yd'] ** 4)
    test['SPEED_SCORE']  = (test['Weight']  * 200.0) / (test['Sprint_40yd']  ** 4)

    vj_mean = train['Vertical_Jump'].mean(); vj_std = train['Vertical_Jump'].std()
    bj_mean = train['Broad_Jump'].mean();    bj_std = train['Broad_Jump'].std()
    train['EXPLOSIVENESS'] = 0.5 * (
        (train['Vertical_Jump'] - vj_mean) / vj_std +
        (train['Broad_Jump']    - bj_mean) / bj_std)
    test['EXPLOSIVENESS'] = 0.5 * (
        (test['Vertical_Jump']  - vj_mean) / vj_std +
        (test['Broad_Jump']     - bj_mean) / bj_std)

    train['AGILITY_RATIO'] = train['Agility_3cone'] / train['Shuttle'].where(train['Shuttle'] > 0, np.nan)
    test['AGILITY_RATIO']  = test['Agility_3cone']  / test['Shuttle'].where(test['Shuttle']  > 0, np.nan)
    ratio_median = train['AGILITY_RATIO'].median()
    train['AGILITY_RATIO'] = train['AGILITY_RATIO'].fillna(ratio_median)
    test['AGILITY_RATIO']  = test['AGILITY_RATIO'].fillna(ratio_median)

    # Label encoding
    for c in CATEGORICAL_COLS:
        categories = train[c].astype(str).unique()
        mapping = {cat: i for i, cat in enumerate(categories)}
        train[c] = train[c].astype(str).map(mapping).astype(int)
        test[c]  = test[c].astype(str).map(mapping).fillna(-1).astype(int)

    drop_cols = [ID_COL, HIGH_CARDINALITY_COL, TARGET]
    feature_cols = [c for c in train.columns if c not in drop_cols]
    X      = train[feature_cols].copy()
    y      = train[TARGET].astype(int)
    X_test = test[feature_cols].copy()
    return X, y, X_test, feature_cols

X, y, X_test, feature_cols = prepare_features(train_raw, test_raw)
print(f'Features ({len(feature_cols)}): {feature_cols}')

Features (25): ['Year', 'Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Player_Type', 'Position_Type', 'Position', 'BMI', 'School_CE', 'Age_missing', 'Sprint_40yd_missing', 'Vertical_Jump_missing', 'Bench_Press_Reps_missing', 'Broad_Jump_missing', 'Agility_3cone_missing', 'Shuttle_missing', 'SPEED_SCORE', 'EXPLOSIVENESS', 'AGILITY_RATIO']


In [9]:
# ── CV helpers ─────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def catboost_cv(params, X, y, X_test=None):
    """CatBoost CV with early stopping (Tutorial 3)."""
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test)) if X_test is not None else None
    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
        va_pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_pred
        fold_auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(fold_auc)
        if X_test is not None:
            test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
        print(f'  Fold {fold}: AUC = {fold_auc:.4f}')
    oof_auc = roc_auc_score(y, oof)
    print(f'  OOF AUC: {oof_auc:.4f}')
    return oof, test_pred, oof_auc, fold_aucs

def cv_fit_predict(make_model, X, y, X_test=None):
    """Generic CV for any sklearn-compatible model (Tutorial 4)."""
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test)) if X_test is not None else None
    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = make_model()
        model.fit(X_tr, y_tr)
        va_pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_pred
        fold_auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(fold_auc)
        if X_test is not None:
            test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
        print(f'  Fold {fold}: AUC = {fold_auc:.4f}')
    oof_auc = roc_auc_score(y, oof)
    print(f'  OOF AUC: {oof_auc:.4f}')
    return oof, test_pred, oof_auc, fold_aucs

In [10]:
# ── Tutorial 3: Optuna-tuned CatBoost ──────────────────────────────────────
# Set FAST_MODE = False for the full 60-trial search (recommended)
FAST_MODE = False
N_TRIALS          = 10  if FAST_MODE else 60
TUNING_ITERATIONS = 600 if FAST_MODE else 1500

print(f'FAST_MODE={FAST_MODE}  N_TRIALS={N_TRIALS}  TUNING_ITERATIONS={TUNING_ITERATIONS}')

fixed_params = {
    'loss_function':         'Logloss',
    'eval_metric':           'AUC',
    'iterations':            TUNING_ITERATIONS,
    'early_stopping_rounds': 100,
    'use_best_model':        True,
    'random_seed':           SEED,
    'verbose':               False,
    'allow_writing_files':   False,
}

# Default baseline first
default_params = {
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'iterations': 1000, 'random_seed': SEED,
    'verbose': False, 'allow_writing_files': False,
}
print('\n--- Default CatBoost ---')
default_oof, default_test_pred, default_auc, default_fold_aucs = catboost_cv(
    default_params, X, y, X_test)
print(f'Default CatBoost OOF AUC: {default_auc:.4f}')

FAST_MODE=False  N_TRIALS=60  TUNING_ITERATIONS=1500

--- Default CatBoost ---
  Fold 1: AUC = 0.8144
  Fold 2: AUC = 0.8614
  Fold 3: AUC = 0.8593
  Fold 4: AUC = 0.8111
  Fold 5: AUC = 0.8470
  OOF AUC: 0.8327
Default CatBoost OOF AUC: 0.8327


In [12]:
# Optuna search
def objective(trial):
    bootstrap_type = trial.suggest_categorical(
        'bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS'])
    params = dict(
        fixed_params,
        depth                      = trial.suggest_int('depth', 4, 10),
        learning_rate              = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        l2_leaf_reg                = trial.suggest_float('l2_leaf_reg', 1e-2, 1e2, log=True),
        random_strength            = trial.suggest_float('random_strength', 0.0, 2.0),
        rsm                        = trial.suggest_float('rsm', 0.5, 1.0),
        border_count               = trial.suggest_int('border_count', 32, 255),
        leaf_estimation_iterations = trial.suggest_int('leaf_estimation_iterations', 1, 10),
        bootstrap_type             = bootstrap_type,
    )
    if bootstrap_type == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 10.0)
    elif bootstrap_type == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
    _, _, auc, _ = catboost_cv(params, X, y, X_test=None)
    return auc

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest OOF AUC : {study.best_value:.4f}')
print(f'Best params  : {study.best_params}')

  0%|          | 0/60 [00:00<?, ?it/s]

  Fold 1: AUC = 0.8071
  Fold 2: AUC = 0.8585
  Fold 3: AUC = 0.8542
  Fold 4: AUC = 0.7902
  Fold 5: AUC = 0.8433
  OOF AUC: 0.7704
  Fold 1: AUC = 0.8060
  Fold 2: AUC = 0.8480
  Fold 3: AUC = 0.8571
  Fold 4: AUC = 0.7838
  Fold 5: AUC = 0.8505
  OOF AUC: 0.7531
  Fold 1: AUC = 0.8050
  Fold 2: AUC = 0.8633
  Fold 3: AUC = 0.8568
  Fold 4: AUC = 0.7858
  Fold 5: AUC = 0.8422
  OOF AUC: 0.7641
  Fold 1: AUC = 0.8104
  Fold 2: AUC = 0.8596
  Fold 3: AUC = 0.8563
  Fold 4: AUC = 0.7825
  Fold 5: AUC = 0.8393
  OOF AUC: 0.7736
  Fold 1: AUC = 0.8111
  Fold 2: AUC = 0.8663
  Fold 3: AUC = 0.8608
  Fold 4: AUC = 0.7890
  Fold 5: AUC = 0.8467
  OOF AUC: 0.7685
  Fold 1: AUC = 0.8040
  Fold 2: AUC = 0.8553
  Fold 3: AUC = 0.8489
  Fold 4: AUC = 0.7892
  Fold 5: AUC = 0.8446
  OOF AUC: 0.7844
  Fold 1: AUC = 0.8098
  Fold 2: AUC = 0.8505
  Fold 3: AUC = 0.8607
  Fold 4: AUC = 0.8021
  Fold 5: AUC = 0.8392
  OOF AUC: 0.8276
  Fold 1: AUC = 0.8124
  Fold 2: AUC = 0.8599
  Fold 3: AUC = 0.8493


In [13]:
# Refit tuned CatBoost
print('--- Tuned CatBoost ---')
best_params = dict(fixed_params, **study.best_params)
tuned_oof, tuned_test_pred, tuned_auc, tuned_fold_aucs = catboost_cv(
    best_params, X, y, X_test)

score_table = pd.DataFrame([
    {'model': 'Default CatBoost', 'OOF AUC': default_auc,
     'fold_std': np.std(default_fold_aucs)},
    {'model': 'Tuned CatBoost',   'OOF AUC': tuned_auc,
     'fold_std': np.std(tuned_fold_aucs)},
]).sort_values('OOF AUC', ascending=False)
print(score_table.to_string(index=False))

# Pick best CatBoost prediction
if tuned_auc >= default_auc:
    cb_test_pred = tuned_test_pred
    cb_oof       = tuned_oof
    print('\nUsing: Tuned CatBoost')
else:
    cb_test_pred = default_test_pred
    cb_oof       = default_oof
    print('\nUsing: Default CatBoost (tuned did not beat default)')

--- Tuned CatBoost ---
  Fold 1: AUC = 0.8158
  Fold 2: AUC = 0.8576
  Fold 3: AUC = 0.8581
  Fold 4: AUC = 0.8042
  Fold 5: AUC = 0.8574
  OOF AUC: 0.8369
           model  OOF AUC  fold_std
  Tuned CatBoost 0.836923  0.023668
Default CatBoost 0.832673  0.021735

Using: Tuned CatBoost


In [14]:
# ── Tutorial 4: Multi-model ensemble ───────────────────────────────────────
def make_adaboost():
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED),
        n_estimators=500, learning_rate=0.05, random_state=SEED)

def make_xgboost():
    return XGBClassifier(
        objective='binary:logistic', eval_metric='auc',
        n_estimators=600, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        reg_lambda=1.0, tree_method='hist', n_jobs=-1, random_state=SEED)

def make_lightgbm():
    return LGBMClassifier(
        objective='binary', n_estimators=1000, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=40, n_jobs=-1, random_state=SEED, verbose=-1)

def make_catboost():
    return CatBoostClassifier(
        loss_function='Logloss', eval_metric='AUC', iterations=1000,
        learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        random_seed=SEED, verbose=False, allow_writing_files=False)

model_factories = {
    'AdaBoost': make_adaboost,
    'XGBoost':  make_xgboost,
    'LightGBM': make_lightgbm,
    'CatBoost': make_catboost,
}

oof_predictions  = {}
test_predictions = {}
model_rows = []

for model_name, make_model in model_factories.items():
    print(f'\n{"="*60}\n{model_name}\n{"="*60}')
    oof, test_pred, auc, fold_aucs = cv_fit_predict(make_model, X, y, X_test)
    oof_predictions[model_name]  = oof
    test_predictions[model_name] = test_pred
    model_rows.append({'model': model_name, 'OOF AUC': auc,
                       'fold_std': np.std(fold_aucs)})

model_comparison = pd.DataFrame(model_rows).sort_values('OOF AUC', ascending=False)
print('\n' + model_comparison.to_string(index=False))


AdaBoost
  Fold 1: AUC = 0.7885
  Fold 2: AUC = 0.8274
  Fold 3: AUC = 0.8501
  Fold 4: AUC = 0.7766
  Fold 5: AUC = 0.8356
  OOF AUC: 0.8125

XGBoost
  Fold 1: AUC = 0.7928
  Fold 2: AUC = 0.8433
  Fold 3: AUC = 0.8246
  Fold 4: AUC = 0.8067
  Fold 5: AUC = 0.8307
  OOF AUC: 0.8195

LightGBM
  Fold 1: AUC = 0.7920
  Fold 2: AUC = 0.8250
  Fold 3: AUC = 0.8021
  Fold 4: AUC = 0.7851
  Fold 5: AUC = 0.8307
  OOF AUC: 0.8066

CatBoost
  Fold 1: AUC = 0.8034
  Fold 2: AUC = 0.8556
  Fold 3: AUC = 0.8232
  Fold 4: AUC = 0.8067
  Fold 5: AUC = 0.8404
  OOF AUC: 0.8248

   model  OOF AUC  fold_std
CatBoost 0.824788  0.019898
 XGBoost 0.819483  0.017858
AdaBoost 0.812549  0.028232
LightGBM 0.806585  0.017966


In [15]:
# Ensemble: equal average + rank average
model_names = list(oof_predictions.keys())
S_oof  = np.column_stack([oof_predictions[m]  for m in model_names])
S_test = np.column_stack([test_predictions[m] for m in model_names])

equal_oof  = S_oof.mean(axis=1)
equal_test = S_test.mean(axis=1)
equal_auc  = roc_auc_score(y, equal_oof)

rank_oof      = np.column_stack([rankdata(S_oof[:,i])  for i in range(S_oof.shape[1])]).mean(axis=1)
rank_test_raw = np.column_stack([rankdata(S_test[:,i]) for i in range(S_test.shape[1])]).mean(axis=1)
rank_test = (rank_test_raw - rank_test_raw.min()) / (rank_test_raw.max() - rank_test_raw.min())
rank_auc  = roc_auc_score(y, rank_oof)

print(f'Equal average ensemble OOF AUC : {equal_auc:.4f}')
print(f'Rank  average ensemble OOF AUC : {rank_auc:.4f}')

Equal average ensemble OOF AUC : 0.8232
Rank  average ensemble OOF AUC : 0.8267


In [ ]:
# ── Select best submission ──────────────────────────────────────────────────
submission_candidates = {**{m: test_predictions[m] for m in model_names},
                         'Equal average ensemble': equal_test,
                         'Rank average ensemble':  rank_test,
                         'Tuned CatBoost (T3)':    cb_test_pred}

oof_candidates = {**{m: oof_predictions[m] for m in model_names},
                  'Equal average ensemble': equal_oof,
                  'Rank average ensemble':  rank_oof,
                  'Tuned CatBoost (T3)':    cb_oof}

score_rows = [{'method': k, 'OOF AUC': roc_auc_score(y, v)}
              for k, v in oof_candidates.items()]
score_table = pd.DataFrame(score_rows).sort_values('OOF AUC', ascending=False)
print(score_table.to_string(index=False))

selected_method = score_table.iloc[0]['method']
test_pred = submission_candidates[selected_method]
print(f'\nSelected: {selected_method}')

                method  OOF AUC
   Tuned CatBoost (T3) 0.836923
 Rank average ensemble 0.826668
              CatBoost 0.824788
Equal average ensemble 0.823207
               XGBoost 0.819483
              AdaBoost 0.812549
              LightGBM 0.806585

Selected: Tuned CatBoost (T3)


In [ ]:
# ── Save submission [DO NOT CHANGE] ────────────────────────────────────────
submission = pd.read_csv(PATH / 'sample_submission.csv')
submission['Drafted'] = test_pred

out_dir = WORK_DIR / 'output'
out_dir.mkdir(exist_ok=True)
submission_path = out_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved to: {submission_path}')
print(submission.head())

Saved to: /content/drive/MyDrive/GCI/competition/output/submission.csv
     Id   Drafted
0  2781  0.609153
1  2782  0.834037
2  2783  0.925729
3  2784  0.911664
4  2785  0.841894


In [ ]:
# ── Auto-download submission.csv ────────────────────────────────────────────
from google.colab import files
files.download(str(submission_path))
print('Download started!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!
